In [2]:
import json
import pandas as pd
import numpy as np
import re
from datetime import datetime

In [46]:
# one = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_ubereats_nonca_ff_03292024.csv"
# two = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_ubereats_nonca_ff_05162024.csv"
# three = "/Users/alyssanguyen/Desktop/IRLE_scraping/scripts/raw_prices_ubereats_nonca_09122024.csv"

# one = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_ubereats_ca_ff_03222024.csv"
# two = "/Users/alyssanguyen/Desktop/IRLE_scraping/scripts/raw_prices_ubereats_ca_09092024.csv"
# three = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_ubereats_ca_ff_05142024.csv"

# one = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_ubereats_ca_fullserv_03252024.csv"
# two = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_ubereats_ca_ffullserv_05142024.csv"
# three = "/Users/alyssanguyen/Desktop/IRLE_scraping/scripts/raw_prices_ubereats_ca_fullserv_09142024.csv"

# one = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_ubereats_nonca_fullserv_03252024.csv"
# two = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_ubereats_nonca_fullserv_05162024.csv"
# three = "/Users/alyssanguyen/Desktop/IRLE_scraping/scripts/raw_prices_ubereats_nonca_fullserv_090142024.csv"

# one = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_ubereats_nonca_fullserv_03252024.csv"
# two = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_ubereats_nonca_fullserv_05162024.csv"
# three = "/Users/alyssanguyen/Desktop/IRLE_scraping/scripts/raw_prices_ubereats_nonca_fullserv_090142024.csv"



# wave_1 = pd.read_csv(one, low_memory = False)
# wave_2 = pd.read_csv(two, low_memory = False)
# wave_3 = pd.read_csv(three, low_memory = False)


file_path_2 = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/uszips.csv"
ca_zip_count = pd.read_csv(file_path_2)

In [50]:
ca_zip_count = ca_zip_count[['zip', 'county_name']]

Data Cleaning 

In [49]:

waves = [wave_1, wave_2, wave_3]   
cleaned_waves = []

for df in waves: 
    #Drop all the columns we don't need 
    ca_ff_ = df.drop(columns=['Unnamed: 0', 'inputted_location','restaurant_distance'])
    ca_ff_ = ca_ff_.dropna(subset=['restaurant_location'])
    ca_zip_count = ca_zip_count[['zip', 'county_name']]

    #restaurant_rating cleaning 
    # Ensure the column is of string type using .loc
    ca_ff_.loc[:, 'restaurant_rating'] = ca_ff_['restaurant_rating'].astype(str)

    # Count rows containing 'mi'
    rows_with_mi = ca_ff_['restaurant_rating'].str.contains('mi').sum()
    print("Number of rows with 'mi' in restaurant rating:", rows_with_mi)

    # Replace invalid ratings ending with 'mi' with '0' using .loc
    ca_ff_.loc[:, 'restaurant_rating'] = ca_ff_['restaurant_rating'].str.replace(r'.*mi$', '0', regex=True)
    ca_ff_ = ca_ff_.dropna()
    ca_ff_ = ca_ff_[ca_ff_['menu_item_price'] != 0]

    #converting data types 
    ca_ff_['restaurant_name'] = ca_ff_['restaurant_name'].astype('string')
    ca_ff_['menu_item'] = ca_ff_['menu_item'].astype('string')
    ca_ff_['menu_item'] = ca_ff_['menu_item'].str.replace(r'\s+', ' ', regex=True)
    ca_ff_['restaurant_location'] = ca_ff_['restaurant_location'].astype('string')
    ca_ff_['restaurant_rating'] = ca_ff_['restaurant_rating'].str.strip().astype(float)

    #cleaning up string columns 

    ca_ff_['menu_item'] = ca_ff_['menu_item'].str.lower()
    ca_ff_['restaurant_location'] = ca_ff_['restaurant_location'].str.lower()
    ca_ff_['restaurant_name'] = ca_ff_['restaurant_name'].str.replace('_', ' ')
    ca_ff_['menu_item'] = ca_ff_['menu_item'].str.replace('®', '', regex=False)  # Remove ® symbol
    #remove special characters
    ca_ff_['menu_item'] = ca_ff_['menu_item'].apply(lambda x: ''.join(ch for ch in x if ch.isalnum() or ch.isspace()))

    cleaned_waves.append(ca_ff_)

KeyError: "['Unnamed: 0', 'inputted_location', 'restaurant_distance'] not found in axis"

In [318]:
def mean_non_zero(x):
    return np.mean(x[x != 0]) if np.any(x != 0) else 0

def median_non_zero(x):
    return np.median(x[x != 0]) if np.any(x != 0) else 0

def std_non_zero(x):
    return np.std(x[x != 0]) if np.any(x != 0) else 0

def price_list(x):
    return list(x)

In [ ]:
#Fast Food Chains Dictionary 
item_dict = {
    "McDonald": {
        "big mac": "specialty_item",
        "cheeseburger": "cheeseburger",
        "hamburger": "hamburger",
        'big mac meal': "combo",
        "medium french fries": "fries",
        "medium diet coke": "drink",
        "apple pie": "dessert"
    },

     "Jack in the Box": {
        "jumbo jack": "specialty_item",
        'large jumbo jack combo': "combo",
        "large french fries": "fries",
        "large cocacola	": "drink",
        "mini churros 5": "dessert"
    },
     "Wendy": {
        "daves single": "specialty_item",
        "jr cheeseburger": "cheeseburger",
        "jr hamburger": "hamburger",
        'daves combo': "combo",
        "french fries": "fries",
        "simply orange juice": "drink",
        "chocolate chunk cookie": "dessert"
    },
     "Burger King": {
        "whopper": "specialty_item",
        "cheeseburger": "cheeseburger",
        "whopper jr": "hamburger",
        'whopper meal': "combo",
        "french fries": "fries",
        "soft drink	": "drink",
        "hersheys sundae pie": "dessert"
        
    },

     "Shake Shack": {
        "shackburger": "specialty_item",
        "cheeseburger": "cheeseburger",
        "hamburger": "hamburger",
        "fries": "fries",
        "fountain soda": "drink",
        "chocolate shake": "dessert"
    },

    "Sonic": {
        "supersonic double cheeseburger'": "specialty_item",
        "quarter pound double cheeseburger": "cheeseburger",
        "supersonic double cheeseburger combo": "combo",
        "fries": "fries",
        "red bull energy drink": "drink",
        "strawberry sundae": "dessert"
    },

      "Five Guys": {
        "cheeseburger": "specialty_item",
        "little cheeseburger": "cheeseburger",
        "little hamburger": "hamburger",
        "regular fries": "fries",
        "simply lemonade": "drink", 
        "milkshake": "dessert"

    },

      "The Habit": {
        "double char": "specialty_item",
       'charburger with cheese': "cheeseburger",
        "charburger": "hamburger",
        '2 original double char meal': "combo",
        "french fries": "fries",
        "regular drink": "drink", 
        "vanilla shake": "dessert"
        
    },
       "Carls Jr": {
        "single big carl": "specialty_item",
       'california classic double cheeseburger': "cheeseburger",
       'single big carl combo': "combo",
        "naturalcut french fries": "fries", 
        "handcrafted lemonade": "drink",
        "handscooped icecream shakes": "dessert"
    },
    
    "Hardee": {
        "famous star": "specialty_item",
       'big cheeseburger': "cheeseburger",
       'famous star combo': "combo",
        "naturalcut french fries": "fries",
        "handcrafted lemonade": "drink",
        "apple turnover": "dessert"
    }

}

In [296]:
rnd_1 = cleaned_waves[0]
restaurant = rnd_1[rnd_1['restaurant_name']== 'Buffalo Wild Wings']
values = restaurant['menu_item'].value_counts().reset_index()
values[values['menu_item'].str.contains("cookie")]['menu_item'].unique()


array(['triple chocolate chip cookie'], dtype=object)

In [319]:
#Full Service Chains Dictionary 

item_dict = {
    "Outback Steakhouse": {
        "the bloomin burger": "specialty_item",
        "boomerang cheeseburger": "cheeseburger",
        "the outbacker burger": "hamburger",
        "coca cola products": "drink",
        "aussie fries": "fries",
        "triplelayer carrot cake": "dessert"
    },

     "Red Robin": {
        "red robin gourmet cheeseburger": "specialty_item",
        "reds cheeseburger": "cheeseburger",
        "steak fries": "fries",
        "soft drinks": "drink",
        "creamy milkshake": "dessert"
    },

     "Denny": {
        "slamburger": "specialty_item",
        "single cheeseburger": "cheeseburger",
        "wavy cut french fries": "fries",
        "soft drinks": "drink",
        "lava cookie skillet": "dessert"
    },

     "Applebee": {
        "neighborhood burger": "specialty_item",
        "classic cheeseburger": "cheeseburger",
        "basket fries": "fries",
        "fountain drinks": "drink",
        "brownie bite": "dessert"
        
    },

     "TGI Fridays": {
        "fridays signature whiskeyglaze burger": "specialty_item",
        "cheeseburger": "cheeseburger",
        "kids classic hamburger": "hamburger",
        "seasoned fries": "fries",
        "diet coke": "drink",
        "cinnabon caramel pecan cheesecake": "dessert"
    },

    "Buffalo Wild Wings": {
        "triple bacon cheeseburger": "specialty_item",
        "allamerican cheeseburger": "cheeseburger",
        "regular french fries": "fries",
        "20oz fountain soda": "drink",
        "triple chocolate chip cookie": "dessert"
    }

}

Data Processing Script 

In [320]:
cleaned_w_prices = []

for df in cleaned_waves: 
    new_data = []

    for restaurant in df["restaurant_name"].unique():
        if restaurant not in item_dict:
            continue

        # Filter dataset to unique restaurant
        restaurant_data = df[df["restaurant_name"] == restaurant]

        # Iterate through each location 
        for location in restaurant_data["restaurant_location"].unique():
            curr_restaurant = restaurant_data[restaurant_data["restaurant_location"] == location]
            
            # Instantiate a new row dictionary
            row = {"restaurant_name": restaurant, "restaurant_location": location}
            
            # Initialize a counter for items priced between $4 and $6
            four_to_six_count = 0
            
            # Count all menu items priced between $4 and $6
            four_to_six_count = curr_restaurant[
                (curr_restaurant["menu_item_price"] >= 4) & (curr_restaurant["menu_item_price"] <= 6)
            ].shape[0]
            
            # For each menu item in the dictionary (combo, hamburger, etc)
            for item, column_name in item_dict[restaurant].items():
                # Find the price of the item
                price = curr_restaurant.loc[curr_restaurant["menu_item"] == item, "menu_item_price"]
                
                # Aggregate that price to the corresponding column 
                row[column_name] = price.iloc[0] if not price.empty else np.nan
            
            # Add the `four_to_six` count to the row
            row["four_to_six"] = four_to_six_count
            
            #Calculate additional metrics
            row["number_of_items"] = curr_restaurant.shape[0]  # Total number of menu items
            row["average_rating"] = curr_restaurant["restaurant_rating"].mean()  # Average rating
            
            #Get the first value of number_of_ratings  
            row["number_of_ratings"] = curr_restaurant["number_of_ratings"].iloc[0]   
            
            row["average_menu_price"] = curr_restaurant["menu_item_price"].mean()  # Average menu price
            
            new_data.append(row)

    result_df = pd.DataFrame(new_data)
    cleaned_w_prices.append(result_df)

In [321]:
cleaned_w_prices[0].head()

,restaurant_name,restaurant_location,specialty_item,cheeseburger,fries,drink,dessert,four_to_six,number_of_items,average_rating,number_of_ratings,average_menu_price,hamburger
0,Applebee,"2409 south mckenzie street, foley, al, 36535, us",15.69,14.99,4.99,2.49,3.19,7,99,3.60,45,12.847576,NaN
1,Applebee,"1250 boots blvd., fultondale, al, 35068, us",13.79,12.69,2.89,3.49,2.29,22,196,4.00,22,11.273673,NaN
2,Applebee,"3150 memorial pkwy nw, huntsville, al, 35810, us",13.79,12.69,2.89,3.49,2.29,10,96,4.00,45,11.513958,NaN
3,Applebee,"160 tanger outlets pkwy., pooler, ga, 31322, us",15.99,14.79,5.39,2.49,2.49,10,200,4.45,74,12.172000,NaN
4,Applebee,"475 franklin rd., marietta, ga, 30067, us",15.69,14.99,3.79,2.49,3.19,9,98,4.10,67,13.164490,NaN


In [322]:
#add wave number
for i in range(len(cleaned_w_prices)):
    wave_no = i + 1 
    cleaned_w_prices[i]['wave'] = wave_no

In [323]:
#add uber_eats bool col 
combined = pd.concat(cleaned_w_prices)
combined['uber_eats'] = 1
#fast food or fullservice 
combined['fast_food'] = 0
#small or large chains 
combined['small'] = 0

In [331]:
idx_lst = [554, 451]
for i in idx_lst: 
    combined.loc[i, 'restaurant_location'] = "9001 shawnee mission pkwy, merriam, ks, 66202"


idx_lst = [595, 510, 877]
for i in idx_lst: 
    combined.loc[i, 'restaurant_location'] = "13635 san pedro ave, san antonio, tx, 78232"

idx_lst = [729, 629, 780]
for i in idx_lst: 
    combined.loc[i, 'restaurant_location'] = "20430 highway 59 n, humble, tx, 77338"

idx_lst = [721, 624]
for i in idx_lst: 
    combined.loc[i, 'restaurant_location'] = "12811 s tryon st, charlotte, nc, 28273"

combined.loc[556, 'restaurant_location'] = "2500 sm 291 hwy, independence, mo, 64055"

In [332]:

pattern = r",\s*([a-zA-Z]{2})\s*,?\s*(\d{5}(?:-\d{4})?)"

def extract_state_zip(address):
    match = re.search(pattern, address)
    if match:
        state, zip_code = match.groups()
        return state, zip_code
    else:
        return None, None

# Apply the function to extract state and zip code
combined[['state', 'zip']] = combined['restaurant_location'].apply(lambda x: pd.Series(extract_state_zip(x)))
combined['zip'] = combined['zip'].str.split('-').str[0].astype(int)

#Get county 
combined = combined.merge(ca_zip_count, on = 'zip')

In [333]:
combined.head()

,restaurant_name,restaurant_location,specialty_item,cheeseburger,fries,drink,dessert,four_to_six,number_of_items,average_rating,...,average_menu_price,hamburger,wave,uber_eats,fast_food,small,state,zip,ca,county_name
0,Applebee,"2409 south mckenzie street, foley, al, 36535, us",15.69,14.99,4.99,2.49,3.19,7,99,3.60,...,12.847576,NaN,1,1,0,0,al,36535,0,Baldwin
1,Applebee,"1250 boots blvd., fultondale, al, 35068, us",13.79,12.69,2.89,3.49,2.29,22,196,4.00,...,11.273673,NaN,1,1,0,0,al,35068,0,Jefferson
2,Applebee,"3150 memorial pkwy nw, huntsville, al, 35810, us",13.79,12.69,2.89,3.49,2.29,10,96,4.00,...,11.513958,NaN,1,1,0,0,al,35810,0,Madison
3,Applebee,"160 tanger outlets pkwy., pooler, ga, 31322, us",15.99,14.79,5.39,2.49,2.49,10,200,4.45,...,12.172000,NaN,1,1,0,0,ga,31322,0,Chatham
4,Applebee,"475 franklin rd., marietta, ga, 30067, us",15.69,14.99,3.79,2.49,3.19,9,98,4.10,...,13.164490,NaN,1,1,0,0,ga,30067,0,Cobb


In [334]:
combined['ca'] = (combined['state'] == 'ca').astype(int)

In [335]:
#Run to check if there are faulty address strings 
nan_zip_rows = combined[combined['zip'].isna()]
nan_zip_rows

,restaurant_name,restaurant_location,specialty_item,cheeseburger,fries,drink,dessert,four_to_six,number_of_items,average_rating,...,average_menu_price,hamburger,wave,uber_eats,fast_food,small,state,zip,ca,county_name


In [336]:
combined.to_csv('ubereats_fullserv_nonca_combined.csv', index = True)

Burger King

In [229]:
one = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_burgerking_ca_03282024.csv"
two = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_burgerking_ca_05152024.csv"
three = "/Users/alyssanguyen/Desktop/IRLE_scraping/scripts/burger_king_ca_09232024.csv"

wave_1 = pd.read_csv(one, low_memory = False)
wave_2 = pd.read_csv(two, low_memory = False)
wave_3 = pd.read_csv(three, low_memory = False)


In [231]:
waves = [wave_1, wave_2, wave_3] 
cleaned_waves = []
idx = 0

for wave in waves:
    wave['restaurant_name'] = 'Burger King'

    wave['menu_item'] = wave['menu_item'].astype('string')
    wave['menu_item'] = wave['menu_item'].str.replace(r'\s+', ' ', regex=True)

    wave['restaurant_address'] = wave['restaurant_address'].astype('string')
    wave['menu_item'] = wave['menu_item'].str.lower()
    wave['restaurant_address'] = wave['restaurant_address'].str.lower()
    wave['menu_item'] = wave['menu_item'].str.replace('®', '', regex=False)  # Remove ® symbol
    #remove special characters
    wave['menu_item'] = wave['menu_item'].apply(lambda x: ''.join(ch for ch in x if ch.isalnum() or ch.isspace()))

    wave["restaurant_address"] = wave["restaurant_address"].str.lower()
    
    if idx == 2: 
        # Extract the first three phrases (words) from input_address
        wave["input_first_three_phrases"] = wave["input_address"].apply(
            lambda x: ' '.join(x.lower().split()[:3]).replace(',', '')
        )
        # Filter rows where input_first_three_phrases matches restaurant_address
        wave = wave[wave["input_first_three_phrases"] == wave["restaurant_address"]]
        wave = wave.drop("restaurant_address", axis = 1)
        wave = wave.rename(columns = {"input_address" : "restaurant_address"})
        wave['menu_item_price'] = wave['menu_item_price'].str.replace('$', '', regex=False)  # Remove ® symbol
        wave['menu_item_price'] = wave['menu_item_price'].astype(float)

    idx += 1 
    cleaned_waves.append(wave)


In [232]:
item_dict = {"Burger King": {
    "whopper": "specialty_item",
    "cheeseburger": "cheeseburger",
    "whopper jr": "hamburger",
    'whopper meal': "combo",
    "french fries": "fries",
    "soft drink	": "drink",
    "hersheys sundae pie": "dessert"
        
    }
}


cleaned_w_prices = []

for wave in cleaned_waves: 
    new_data = []
    for restaurant in wave["restaurant_name"].unique():
        if restaurant not in item_dict:
            continue

        # Filter dataset to unique restaurant
        restaurant_data = wave[wave["restaurant_name"] == restaurant]

        # Iterate through each location 
        for location in restaurant_data["restaurant_address"].unique():
            curr_restaurant = restaurant_data[restaurant_data["restaurant_address"] == location]
                
            # Instantiate a new row dictionary
            row = {"restaurant_name": restaurant, "restaurant_address": location}
                
            # Initialize a counter for items priced between $4 and $6
            four_to_six_count = 0
                
            # Count all menu items priced between $4 and $6
            four_to_six_count = curr_restaurant[
                (curr_restaurant["menu_item_price"] >= 4) & (curr_restaurant["menu_item_price"] <= 6)
            ].shape[0]
                
            # For each menu item in the dictionary (combo, hamburger, etc)
            for item, column_name in item_dict[restaurant].items():
                # Find the price of the item
                price = curr_restaurant.loc[curr_restaurant["menu_item"] == item, "menu_item_price"]
                    
                # Aggregate that price to the corresponding column 
                row[column_name] = price.iloc[0] if not price.empty else np.nan
                
            # Add the `four_to_six` count to the row
            row["four_to_six"] = four_to_six_count
                
            new_data.append(row)
    result_df = pd.DataFrame(new_data)
    cleaned_w_prices.append(result_df)


In [233]:
#add wave number
for i in range(len(cleaned_w_prices)):
    wave_no = i + 1 
    cleaned_w_prices[i]['wave'] = wave_no

In [234]:
#add uber_eats bool col 
combined = pd.concat(cleaned_w_prices)
combined['uber_eats'] = 0
#fast food or fullservice 
combined['fast_food'] = 1
#small or large chains 
combined['small'] = 0
combined = combined.rename(columns = {"restaurant_address": "restaurant_location"})

In [235]:
pattern = r",\s*([a-zA-Z]{2})\s*,?\s*(\d{5}(?:-\d{4})?)"

def extract_state_zip(address):
    match = re.search(pattern, address)
    if match:
        state, zip_code = match.groups()
        return state, zip_code
    else:
        return None, None

# Apply the function to extract state and zip code
combined[['state', 'zip']] = combined['restaurant_location'].apply(lambda x: pd.Series(extract_state_zip(x)))
combined['zip'] = combined['zip'].str.split('-').str[0].astype(int)

#Get county 
combined = combined.merge(ca_zip_count, on = 'zip')

combined['ca'] = (combined['state'] == 'ca').astype(int)

In [225]:
nan_zip_rows = combined[combined['zip'].isna()]
nan_zip_rows

,restaurant_name,restaurant_location,specialty_item,cheeseburger,hamburger,combo,fries,drink,dessert,four_to_six,wave,uber_eats,fast_food,small,state,zip,county_name,ca


In [236]:
combined[['number_of_items', 'average_rating', 'number_of_ratings', 'average_menu_price']] = np.nan

In [237]:
mega = pd.read_csv("ubereats_mega_combined.csv")

In [238]:
mega_combined = pd.concat([combined, mega])
mega_combined.to_csv('ubereats_mega_combined.csv', index = True)


Wendy's 

In [254]:
one = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_wendys_nonca_03302024.csv"
two = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_wendys_nonca_05142024.csv"
three = "/Users/alyssanguyen/Desktop/IRLE_scraping/scripts/wendy_nonca_09222014.csv"

wave_1 = pd.read_csv(one, low_memory = False)
wave_2 = pd.read_csv(two, low_memory = False)
wave_3 = pd.read_csv(three, low_memory = False)

In [255]:
wave_1 = wave_1.rename(columns={"price": "menu_item_price", "address" : "restaurant_address"})
wave_2 = wave_2.rename(columns={"price": "menu_item_price", "address" : "restaurant_address"})
wave_1 = wave_1.drop(["Unnamed: 0", "calories"], axis = 1)
wave_2 = wave_2.drop(["Unnamed: 0", "calories"], axis = 1)
wave_3 = wave_3.drop(["menu_item_calories"], axis = 1)

In [256]:
waves = [wave_1, wave_2, wave_3] 
cleaned_waves = []

for wave in waves:
    wave = wave[wave['menu_item_price'] != "."]
    wave['menu_item_price'] = wave['menu_item_price'].str.replace("$", "")

    # # Define a function to split prices by "-" or "/"
    def split_and_average(price_str):
        # # Split the price string by "-" or "/"
        prices = re.split(r'[-/]', price_str)
        # # Convert prices to float and calculate the average
        #prices = price_str.replace("$", "")
        prices = [float(price) for price in prices]
        return sum(prices) / len(prices)
        
    wave['menu_item_price'] = wave['menu_item_price'].apply(split_and_average)
    wave['restaurant_name'] = 'Wendy'

    wave['menu_item'] = wave['menu_item'].astype('string')
    wave['menu_item'] = wave['menu_item'].str.replace(r'\s+', ' ', regex=True)

    wave['restaurant_address'] = wave['restaurant_address'].astype('string')
    wave['menu_item'] = wave['menu_item'].str.lower()
    wave['restaurant_address'] = wave['restaurant_address'].str.lower()
    wave['menu_item'] = wave['menu_item'].str.replace('®', '', regex=False)  # Remove ® symbol
    #remove special characters
    wave['menu_item'] = wave['menu_item'].apply(lambda x: ''.join(ch for ch in x if ch.isalnum() or ch.isspace()))

    wave["restaurant_address"] = wave["restaurant_address"].str.lower()
    wave['menu_item_price'] = wave['menu_item_price'].astype(float)

    cleaned_waves.append(wave)


In [257]:
item_dict = {"Wendy": {
        "daves single": "specialty_item",
        "jr cheeseburger": "cheeseburger",
        "jr hamburger": "hamburger",
        'daves combo': "combo",
        "french fries": "fries",
        "simply orange juice": "drink",
        "chocolate chunk cookie": "dessert"
    }
}

cleaned_w_prices = []

for wave in cleaned_waves: 
    new_data = []
    for restaurant in wave["restaurant_name"].unique():
        # Filter dataset to unique restaurant
        restaurant_data = wave[wave["restaurant_name"] == restaurant]

        # Iterate through each location 
        for location in restaurant_data["restaurant_address"].unique():
            curr_restaurant = restaurant_data[restaurant_data["restaurant_address"] == location]
                
            # Instantiate a new row dictionary
            row = {"restaurant_name": restaurant, "restaurant_address": location}
                
            # Initialize a counter for items priced between $4 and $6
            four_to_six_count = 0
                
            # Count all menu items priced between $4 and $6
            four_to_six_count = curr_restaurant[
                (curr_restaurant["menu_item_price"] >= 4) & (curr_restaurant["menu_item_price"] <= 6)
            ].shape[0]
                
            # For each menu item in the dictionary (combo, hamburger, etc)
            for item, column_name in item_dict[restaurant].items():
                # Find the price of the item
                price = curr_restaurant.loc[curr_restaurant["menu_item"] == item, "menu_item_price"]
                    
                # Aggregate that price to the corresponding column 
                row[column_name] = price.iloc[0] if not price.empty else np.nan
                
            # Add the `four_to_six` count to the row
            row["four_to_six"] = four_to_six_count
                
            new_data.append(row)
    result_df = pd.DataFrame(new_data)
    cleaned_w_prices.append(result_df)

In [258]:
#add wave number
for i in range(len(cleaned_w_prices)):
    wave_no = i + 1 
    cleaned_w_prices[i]['wave'] = wave_no

In [259]:
#add uber_eats bool col 
combined = pd.concat(cleaned_w_prices)
combined['uber_eats'] = 0
#fast food or fullservice 
combined['fast_food'] = 1
#small or large chains 
combined['small'] = 0
combined = combined.rename(columns = {"restaurant_address": "restaurant_location"})

In [260]:
pattern = r",\s*([a-zA-Z]{2})\s*,?\s*(\d{5}(?:-\d{4})?)"

def extract_state_zip(address):
    match = re.search(pattern, address)
    if match:
        state, zip_code = match.groups()
        return state, zip_code
    else:
        return None, None

# Apply the function to extract state and zip code
combined[['state', 'zip']] = combined['restaurant_location'].apply(lambda x: pd.Series(extract_state_zip(x)))
combined['zip'] = combined['zip'].str.split('-').str[0].astype(int)

#Get county 
combined = combined.merge(ca_zip_count, on = 'zip')

combined['ca'] = (combined['state'] == 'ca').astype(int)

In [262]:
combined[['number_of_items', 'average_rating', 'number_of_ratings', 'average_menu_price']] = np.nan

In [263]:
len(combined.columns)

22

In [266]:
mega = pd.read_csv("ubereats_mega_combined.csv")

In [265]:
mega_combined = pd.concat([combined, mega])
mega_combined.to_csv('ubereats_mega_combined.csv', index = True)

Carl's Jr. 

In [302]:
one = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_hardees_non_ca_03292024.csv"
two = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/raw_prices_hardees_non_ca_05162024.csv"
three = "/Users/alyssanguyen/Desktop/IRLE_scraping/scripts/raw_prices_hardees_nonca_10102024.csv"

wave_1 = pd.read_csv(one, low_memory = False)
wave_2 = pd.read_csv(two, low_memory = False)
wave_3 = pd.read_csv(three, low_memory = False)

In [307]:
waves = [wave_1, wave_2, wave_3] 
cleaned_waves = []
idx = 0

for wave in waves:
    wave['restaurant_name'] = 'Hardee'

    wave['menu_item'] = wave['menu_item'].astype('string')
    wave['menu_item'] = wave['menu_item'].str.replace(r'\s+', ' ', regex=True)

    wave['restaurant_address'] = wave['restaurant_address'].astype('string')
    wave['menu_item'] = wave['menu_item'].str.lower()
    wave['restaurant_address'] = wave['restaurant_address'].str.lower()
    wave['menu_item'] = wave['menu_item'].str.replace('®', '', regex=False)  # Remove ® symbol
    #remove special characters
    wave['menu_item'] = wave['menu_item'].apply(lambda x: ''.join(ch for ch in x if ch.isalnum() or ch.isspace()))

    wave["restaurant_address"] = wave["restaurant_address"].str.lower()
    cleaned_waves.append(wave)


In [308]:
item_dict = { "Hardee": {
        "famous star": "specialty_item",
       'big cheeseburger': "cheeseburger",
       'famous star combo': "combo",
        "naturalcut french fries": "fries",
        "handcrafted lemonade": "drink",
        "apple turnover": "dessert"
    }

}


cleaned_w_prices = []

for wave in cleaned_waves: 
    new_data = []
    for restaurant in wave["restaurant_name"].unique():
        if restaurant not in item_dict:
            continue

        # Filter dataset to unique restaurant
        restaurant_data = wave[wave["restaurant_name"] == restaurant]

        # Iterate through each location 
        for location in restaurant_data["restaurant_address"].unique():
            curr_restaurant = restaurant_data[restaurant_data["restaurant_address"] == location]
                
            # Instantiate a new row dictionary
            row = {"restaurant_name": restaurant, "restaurant_address": location}
                
            # Initialize a counter for items priced between $4 and $6
            four_to_six_count = 0
                
            # Count all menu items priced between $4 and $6
            four_to_six_count = curr_restaurant[
                (curr_restaurant["menu_item_price"] >= 4) & (curr_restaurant["menu_item_price"] <= 6)
            ].shape[0]
                
            # For each menu item in the dictionary (combo, hamburger, etc)
            for item, column_name in item_dict[restaurant].items():
                # Find the price of the item
                price = curr_restaurant.loc[curr_restaurant["menu_item"] == item, "menu_item_price"]
                    
                # Aggregate that price to the corresponding column 
                row[column_name] = price.iloc[0] if not price.empty else np.nan
                
            # Add the `four_to_six` count to the row
            row["four_to_six"] = four_to_six_count
                
            new_data.append(row)
    result_df = pd.DataFrame(new_data)
    cleaned_w_prices.append(result_df)


In [309]:
#add wave number
for i in range(len(cleaned_w_prices)):
    wave_no = i + 1 
    cleaned_w_prices[i]['wave'] = wave_no

In [310]:
#add uber_eats bool col 
combined = pd.concat(cleaned_w_prices)
combined['uber_eats'] = 0
#fast food or fullservice 
combined['fast_food'] = 1
#small or large chains 
combined['small'] = 0
combined = combined.rename(columns = {"restaurant_address": "restaurant_location"})

In [311]:
pattern = r",\s*([a-zA-Z]{2})\s*,?\s*(\d{5}(?:-\d{4})?)"

def extract_state_zip(address):
    match = re.search(pattern, address)
    if match:
        state, zip_code = match.groups()
        return state, zip_code
    else:
        return None, None

# Apply the function to extract state and zip code
combined[['state', 'zip']] = combined['restaurant_location'].apply(lambda x: pd.Series(extract_state_zip(x)))
combined['zip'] = combined['zip'].str.split('-').str[0].astype(int)

#Get county 
combined = combined.merge(ca_zip_count, on = 'zip')

combined['ca'] = (combined['state'] == 'ca').astype(int)

In [312]:
combined.head()

,restaurant_name,restaurant_location,specialty_item,cheeseburger,combo,fries,drink,dessert,four_to_six,wave,uber_eats,fast_food,small,state,zip,county_name,ca
0,Hardee,"6671 roswell rd ne, sandy springs, ga, 30328, us",NaN,5.09,NaN,NaN,NaN,NaN,2,1,0,1,0,ga,30328,Fulton,0
1,Hardee,"161 marietta hwy, canton, ga, 30114, us",NaN,5.09,NaN,NaN,NaN,NaN,2,1,0,1,0,ga,30114,Cherokee,0
2,Hardee,"1097 highway 92, acworth, ga, 30102, us",NaN,5.09,NaN,NaN,NaN,NaN,2,1,0,1,0,ga,30102,Cherokee,0
3,Hardee,"4850 floyd road sw, mableton, ga, 30126, us",NaN,5.09,NaN,NaN,NaN,NaN,2,1,0,1,0,ga,30126,Cobb,0
4,Hardee,"125 w maple st, cumming, ga, 30040, us",5.89,5.09,NaN,NaN,NaN,NaN,3,1,0,1,0,ga,30040,Forsyth,0


In [313]:
mega = pd.read_csv("ubereats_mega_combined.csv")

In [314]:
mega_combined = pd.concat([combined, mega])
mega_combined.to_csv('ubereats_mega_combined.csv', index = True)

In [319]:
mega_combined = mega_combined.loc[:, ~mega_combined.columns.str.contains('Unnamed')]
mega_combined.to_csv('ubereats_mega_combined.csv', index = True)

Attrition Shit

In [267]:
wave_3 = pd.read_csv("/Users/alyssanguyen/Desktop/IRLE_scraping/scripts/ubereats_ca_ff_w3.csv")

scraped = pd.read_csv(
    "/Users/alyssanguyen/Desktop/IRLE_scraping/scripts/simplified_scrape_results.csv",
)


In [268]:
import pandas as pd
import json

# Load the JSON file
with open("simplified_scrape_results_1.json", "r") as file:
    data = json.load(file)

# Normalize the JSON data into a flat table
df = pd.json_normalize(data)

# Export the normalized DataFrame to a CSV file
csv_file_path = "simplified_scrape_results_1.csv"
df.to_csv(csv_file_path, index=False)

print(f"Nested JSON data has been normalized and saved as CSV at {csv_file_path}")


Nested JSON data has been normalized and saved as CSV at simplified_scrape_results_1.csv


In [269]:
scraped_1 = pd.read_csv("simplified_scrape_results_1.csv")
scraped_1['street'] = scraped_1['location'].str.split(",").str[0]
scraped_1['street_uber'] = scraped_1['address'].str.split(",").str[0]
scraped_1['street_uber'] = scraped_1['street_uber'].str.lower()
scraped_1

,location,restaurant,address,street,street_uber
0,"2200 otis drive, alameda, ca, 94501, us",Burger King,"2200 Otis Drive, ALAMEDA, CA 94501-5730",2200 otis drive,2200 otis drive
1,"620 w foothill blvd, rialto, ca, 92376, us",Carls Jr,"620 W Foothill Blvd, Rialto, CA 92376",620 w foothill blvd,620 w foothill blvd
2,"172 bernal rd, san jose, ca, 95139, us",Carls Jr,"172 Bernal Rd, San Jose, CA 95139",172 bernal rd,172 bernal rd
3,"516 north beaudry avenue, los angeles, ca, 900...",Jack in the Box,"516 North Beaudry Avenue, Los Angeles, CA 90012",516 north beaudry avenue,516 north beaudry avenue
4,"24820 pico canyon road, santa clarita, ca, 913...",Jack in the Box,"24820 Pico Canyon Road, Santa Clarita, CA 91381",24820 pico canyon road,24820 pico canyon road
5,"18955 west soledad cyn road, canyon country, c...",Jack in the Box,"18525 Via Princessa, Santa Clarita, CA 91387",18955 west soledad cyn road,18525 via princessa
6,"3521 central ave, riverside, ca, 92506, us",Jack in the Box,"3521 Central Ave, Riverside, CA 92506",3521 central ave,3521 central ave
7,"12440 amargosa rd., victorville, ca, 92392, us",Jack in the Box,"12440 Amargosa Rd., Victorville, CA 92392",12440 amargosa rd.,12440 amargosa rd.
8,"1778 e. shaw, fresno, ca, 93710, us",Wendy,"1778 E. Shaw, Fresno, CA 93710",1778 e. shaw,1778 e. shaw
9,"8600 curbaril ave, atascadero, ca, 93422, us",Wendy,"8600 Curbaril Ave, Atascadero, CA 93422",8600 curbaril ave,8600 curbaril ave


In [270]:
scraped = scraped[['street', 'city', 'state', 'zip', 'restaurant', 'street_uber']]
short_addresses = scraped['street']

In [271]:
joined = pd.concat([scraped, scraped_1])

In [272]:
wave_3['restaurant_shortened'] = wave_3['restaurant_location'].apply(lambda x: ", ".join(x.split(", ")[:1]))

In [273]:
joined[joined['street'] == joined['street_uber']]
short_addresses = joined['street']

In [274]:
missing = wave_3[~wave_3['restaurant_shortened'].isin(short_addresses)]

In [286]:
len(missing)

29

In [275]:
missing[missing['county_name'] == 'Los Angeles'][['restaurant_name', 'restaurant_location']]

,restaurant_name,restaurant_location
749,Burger King,"700 east cesar e chavez avenue, los angeles, c..."
990,Shake Shack,"3525 west carson street, space vc 08, torrance..."
1058,Five Guys,"735 s. figueroa st., #120, los angeles, ca, 90..."
1059,Five Guys,"530 w 27th st. #101, los angeles, ca, 90007, us"
1063,Five Guys,"south bay pavilion 20700 s avalon blvd, carson..."
1068,Five Guys,"12930 ventura blvd., suite 106, studio city, c..."
1512,The Habit,"10370 sepulveda blvd, mission hills, usa, los ..."


In [276]:
missing.to_csv("attrition_1.csv")

In [263]:
zips = [91001, 91003, 91004, 90272]
# missing[missing['county_name'] == "Los Angeles"]

In [277]:
missing_ = missing['restaurant_name'].value_counts().reset_index()
full = wave_3['restaurant_name'].value_counts().reset_index()
full = full.merge(missing_, left_on = "restaurant_name", right_on = "restaurant_name")

In [278]:
full["% missing"] = full["count_y"] / full["count_x"] * 100

In [281]:
full = full.rename({"count_y": "missing_locations", "count_x": "total_locations"}, axis = 1)

In [287]:
full

,restaurant_name,total_locations,missing_locations,% missing
0,McDonald,276,6,2.173913
1,Jack in the Box,248,1,0.403226
2,Burger King,227,2,0.881057
3,Carls Jr,217,1,0.460829
4,Wendy,192,2,1.041667
5,The Habit,139,2,1.438849
6,Five Guys,118,13,11.016949
7,Shake Shack,51,2,3.921569
